In [26]:
!pip install faiss-cpu langchain-community langchain-text-splitters langchain-openai openai


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: C:\Users\lenov\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [27]:
import os
import numpy as np
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

stream_handler = StreamingStdOutCallbackHandler()

llm = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o",
    streaming=True,
    callbacks=[stream_handler],
    request_timeout=600,
    temperature=0.1
)

client = OpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
)

embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    openai_api_key=os.getenv("GITHUB_TOKEN"),
)

print("✓ Modelo configurado con streaming habilitado")
print(f"Modelo: {llm.model_name}")
print(f"Streaming: {llm.streaming}")

✓ Modelo configurado con streaming habilitado
Modelo: gpt-4o
Streaming: True


In [28]:
conocimiento_camanchaca = (
    "Salmones Camanchaca opera centros de cultivo de Salmón Atlántico (Salmo salar) "
    "en la región de Los Lagos, Chile, principalmente en las localidades de Ensenada, Puelche y Huito. "
    "El ciclo productivo del salmón Atlántico comprende tres etapas: smoltificación en agua dulce, "
    "engorda en agua de mar y cosecha. La etapa de engorda en mar tiene una duración aproximada de 14 a 18 meses. "
    "El Factor de Conversión Alimenticia (FCR) es uno de los indicadores más importantes en acuicultura. "
    "Para salmón Atlántico en etapa de engorda, un FCR óptimo se encuentra entre 1.1 y 1.3. "
    "Un FCR superior a 1.4 indica ineficiencia alimentaria y puede deberse a sobrealimentación, "
    "temperatura del agua elevada, problemas sanitarios o baja calidad del pellet utilizado. "
    "La mortalidad diaria en centros de engorda no debe superar el 0.5% bajo condiciones normales. "
    "Una mortalidad entre 0.5% y 1.5% se considera alerta temprana y requiere investigación. "
    "Una mortalidad superior al 2% es crítica e indica presencia de enfermedades o problemas de manejo. "
    "Las principales enfermedades que afectan la producción en Los Lagos son: "
    "SRS (Piscirickettsiosis), ISA (Anemia Infecciosa del Salmón), Caligus (piojo de mar) y BKD. "
    "El oxígeno disuelto en las jaulas debe mantenerse por encima de 7 mg/L en todo momento. "
    "La temperatura óptima de cultivo para salmón Atlántico es entre 8°C y 14°C. "
    "La biometría se realiza cada 4 semanas para medir el peso promedio por jaula y ajustar raciones. "
    "Una alta desviación estándar entre jaulas indica crecimiento no homogéneo. "
    "El peso de cosecha objetivo para Salmón Atlántico en Camanchaca es de 4.5 kg promedio. "
    "La mortalidad aceptable durante cosecha no debe superar el 1%. "
    "El alimento representa aproximadamente el 50% de los costos operacionales de un centro de cultivo. "
    "Los tratamientos con antibióticos requieren autorización del Sernapesca y deben respetar "
    "los períodos de carencia antes de la cosecha."
)

print("✓ Base de conocimiento técnico Camanchaca cargada")
print(f"Longitud del documento: {len(conocimiento_camanchaca)} caracteres.")

✓ Base de conocimiento técnico Camanchaca cargada
Longitud del documento: 1911 caracteres.


In [29]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
    length_function=len
)

chunks = text_splitter.split_text(conocimiento_camanchaca)

print(f"✓ Documento dividido en {len(chunks)} chunks\n")

for i, chunk in enumerate(chunks):
    print(f"--- CHUNK {i+1} (Longitud: {len(chunk)}) ---")
    print(chunk)
    print()

✓ Documento dividido en 6 chunks

--- CHUNK 1 (Longitud: 392) ---
Salmones Camanchaca opera centros de cultivo de Salmón Atlántico (Salmo salar) en la región de Los Lagos, Chile, principalmente en las localidades de Ensenada, Puelche y Huito. El ciclo productivo del salmón Atlántico comprende tres etapas: smoltificación en agua dulce, engorda en agua de mar y cosecha. La etapa de engorda en mar tiene una duración aproximada de 14 a 18 meses. El Factor de

--- CHUNK 2 (Longitud: 396) ---
una duración aproximada de 14 a 18 meses. El Factor de Conversión Alimenticia (FCR) es uno de los indicadores más importantes en acuicultura. Para salmón Atlántico en etapa de engorda, un FCR óptimo se encuentra entre 1.1 y 1.3. Un FCR superior a 1.4 indica ineficiencia alimentaria y puede deberse a sobrealimentación, temperatura del agua elevada, problemas sanitarios o baja calidad del pellet

--- CHUNK 3 (Longitud: 391) ---
elevada, problemas sanitarios o baja calidad del pellet utilizado. La mortalid

In [30]:
try:
    chunk_embeddings = client.embeddings.create(
        model="text-embedding-3-small",
        input=chunks
    )
    embeddings_list = [item.embedding for item in chunk_embeddings.data]
    print(f"✓ Se generaron {len(embeddings_list)} embeddings.")
    print(f"Dimensión de cada embedding: {len(embeddings_list[0])}")
    print(f"\nEjemplo - Chunk 1:\n{chunks[0]}")
    print(f"\nEmbedding (primeros 10 de {len(embeddings_list[0])} dimensiones):")
    print(embeddings_list[0][:10])
except Exception as e:
    print(f"Error al generar embeddings: {e}")

✓ Se generaron 6 embeddings.
Dimensión de cada embedding: 1536

Ejemplo - Chunk 1:
Salmones Camanchaca opera centros de cultivo de Salmón Atlántico (Salmo salar) en la región de Los Lagos, Chile, principalmente en las localidades de Ensenada, Puelche y Huito. El ciclo productivo del salmón Atlántico comprende tres etapas: smoltificación en agua dulce, engorda en agua de mar y cosecha. La etapa de engorda en mar tiene una duración aproximada de 14 a 18 meses. El Factor de

Embedding (primeros 10 de 1536 dimensiones):
[0.051318343728780746, 0.020690949633717537, 0.06907624751329422, 0.025798840448260307, 0.03711201995611191, 0.03844885155558586, -0.0022721136920154095, -0.016690433025360107, -0.03781036660075188, -0.06149422004818916]


In [31]:
try:
    vector_db = FAISS.from_texts(texts=chunks, embedding=embeddings_model)
    print("✓ Base de datos vectorial FAISS creada con conocimiento técnico de Camanchaca.")
except Exception as e:
    print(f"Error al crear base vectorial: {e}")

✓ Base de datos vectorial FAISS creada con conocimiento técnico de Camanchaca.


In [32]:
consultas_prueba = [
    "¿Cuál es el FCR aceptable para salmón Atlántico?",
    "Tenemos muchos peces muertos esta semana, ¿qué hacemos?",
    "¿Cuánto debe pesar el salmón al momento de cosechar?",
]

retriever = vector_db.as_retriever(search_kwargs={"k": 2})

print("=== PRUEBA DE RECUPERACIÓN SEMÁNTICA ===\n")

for consulta in consultas_prueba:
    print(f"Consulta: '{consulta}'")
    docs_relevantes = retriever.invoke(consulta)
    for i, doc in enumerate(docs_relevantes):
        print(f"  Chunk relevante {i+1}: {doc.page_content[:120]}...")
    print()

=== PRUEBA DE RECUPERACIÓN SEMÁNTICA ===

Consulta: '¿Cuál es el FCR aceptable para salmón Atlántico?'
  Chunk relevante 1: una duración aproximada de 14 a 18 meses. El Factor de Conversión Alimenticia (FCR) es uno de los indicadores más import...
  Chunk relevante 2: La biometría se realiza cada 4 semanas para medir el peso promedio por jaula y ajustar raciones. Una alta desviación est...

Consulta: 'Tenemos muchos peces muertos esta semana, ¿qué hacemos?'
  Chunk relevante 1: de manejo. Las principales enfermedades que afectan la producción en Los Lagos son: SRS (Piscirickettsiosis), ISA (Anemi...
  Chunk relevante 2: La biometría se realiza cada 4 semanas para medir el peso promedio por jaula y ajustar raciones. Una alta desviación est...

Consulta: '¿Cuánto debe pesar el salmón al momento de cosechar?'
  Chunk relevante 1: La biometría se realiza cada 4 semanas para medir el peso promedio por jaula y ajustar raciones. Una alta desviación est...
  Chunk relevante 2: una duración apr

In [33]:
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """Eres un asistente experto en acuicultura para Salmones Camanchaca.
Tu objetivo es apoyar al equipo con análisis de producción, salud de peces y logística
en los centros de cultivo de la región de Los Lagos. Responde de forma técnica y precisa.
Usa el siguiente contexto para fundamentar tus respuestas:

{context}"""),
    ("human", "{question}")
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("✓ Pipeline RAG inicializado para Salmones Camanchaca.\n")

✓ Pipeline RAG inicializado para Salmones Camanchaca.



In [34]:
print("=== CASO 1: Evaluación de FCR en centro Huito ===\n")
query_fcr = "El centro Huito tiene un FCR de 1.45 esta semana. ¿Es eficiente y qué medidas recomiendas?"
print(f"Consulta: {query_fcr}\n")
print("Respuesta RAG:")
response_fcr = qa_chain.invoke(query_fcr)
print(f"\nFuente: Base de conocimiento técnico Camanchaca\n")

=== CASO 1: Evaluación de FCR en centro Huito ===

Consulta: El centro Huito tiene un FCR de 1.45 esta semana. ¿Es eficiente y qué medidas recomiendas?

Respuesta RAG:
Un FCR de 1.45 en el centro Huito está por encima del rango óptimo para la etapa de engorda de salmón Atlántico (1.1-1.3) y, por lo tanto, indica ineficiencia alimentaria. Esto puede tener un impacto negativo en los costos operacionales, ya que el alimento representa aproximadamente el 50% de estos costos. Además, un FCR elevado puede ser un indicador de problemas subyacentes que deben abordarse.

### Posibles causas y medidas recomendadas:

1. **Sobrealimentación:**
   - **Causa:** Si se está suministrando más alimento del necesario, los peces no lo consumen completamente, lo que aumenta el FCR.
   - **Medida:** Revisar los protocolos de alimentación y ajustar las raciones basándose en los datos de biometría más recientes. Asegurarse de que las raciones estén alineadas con el peso promedio de los peces y las condiciones

In [35]:
print("=== CASO 2: Mortalidad elevada en centro Ensenada ===\n")
query_mort = "En el centro Ensenada llevamos 3 días con mortalidad del 2.5%. ¿Qué protocolo debemos seguir?"
print(f"Consulta: {query_mort}\n")
print("Respuesta RAG:")
response_mort = qa_chain.invoke(query_mort)
print(f"\nFuente: Base de conocimiento técnico Camanchaca\n")

=== CASO 2: Mortalidad elevada en centro Ensenada ===

Consulta: En el centro Ensenada llevamos 3 días con mortalidad del 2.5%. ¿Qué protocolo debemos seguir?

Respuesta RAG:
Una mortalidad diaria del 2.5% durante tres días consecutivos en el centro de cultivo Ensenada es una situación crítica que requiere una respuesta inmediata. Según el contexto proporcionado, esta mortalidad elevada puede estar asociada a enfermedades, problemas de manejo o factores ambientales adversos. A continuación, detallo el protocolo que se debe seguir:

### 1. **Notificación inmediata**
   - Informar de inmediato al equipo técnico y a la gerencia de producción sobre la situación.
   - Notificar a Sernapesca, ya que una mortalidad superior al 2% es considerada crítica y puede estar asociada a enfermedades de notificación obligatoria.

### 2. **Diagnóstico sanitario**
   - **Toma de muestras:** Recoger muestras representativas de peces muertos y moribundos para análisis de laboratorio. Estas muestras deben in

In [36]:
print("=== CASO 3: Evaluación post-cosecha centro Ensenada ===\n")
query_cosecha = "Cosechamos 5000 peces con peso promedio 4.5kg y 2% de mortalidad durante la cosecha. ¿Cómo evalúas estos resultados según los estándares de Camanchaca?"
print(f"Consulta: {query_cosecha}\n")
print("Respuesta RAG:")
response_cosecha = qa_chain.invoke(query_cosecha)
print(f"\nFuente: Base de conocimiento técnico Camanchaca\n")

=== CASO 3: Evaluación post-cosecha centro Ensenada ===

Consulta: Cosechamos 5000 peces con peso promedio 4.5kg y 2% de mortalidad durante la cosecha. ¿Cómo evalúas estos resultados según los estándares de Camanchaca?

Respuesta RAG:
Según los estándares de Salmones Camanchaca, los resultados de esta cosecha presentan un desempeño mixto:

1. **Peso promedio de cosecha**: El peso promedio de 4.5 kg cumple con el objetivo establecido para el Salmón Atlántico. Esto indica que el manejo nutricional y las condiciones de cultivo fueron adecuadas para alcanzar el peso deseado.

2. **Mortalidad durante la cosecha**: La mortalidad del 2% durante la cosecha excede el límite aceptable del 1% establecido por los estándares de Camanchaca. Esto es una señal de alerta que requiere una revisión detallada de los procedimientos de cosecha. Las posibles causas de esta mortalidad elevada pueden incluir:
   - Estrés excesivo durante la manipulación o el transporte.
   - Problemas con la calidad del agua e

In [37]:
def responder_sin_rag(client, pregunta):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "Eres un asistente de acuicultura general."},
            {"role": "user", "content": pregunta}
        ],
        temperature=0.1,
        max_tokens=300
    )
    return response.choices[0].message.content.strip()

pregunta_comparacion = "¿Cuál es el peso de cosecha objetivo para el salmón Atlántico en Camanchaca?"

print("=== COMPARACIÓN: CON RAG vs. SIN RAG ===\n")

print("--- SIN RAG (solo conocimiento general del LLM) ---")
respuesta_sin_rag = responder_sin_rag(client, pregunta_comparacion)
print(respuesta_sin_rag)

print("\n--- CON RAG (usando base de conocimiento de Camanchaca) ---")
print(f"Consulta: {pregunta_comparacion}\n")
respuesta_con_rag = qa_chain.invoke(pregunta_comparacion)

=== COMPARACIÓN: CON RAG vs. SIN RAG ===

--- SIN RAG (solo conocimiento general del LLM) ---
El peso de cosecha objetivo para el salmón Atlántico (Salmo salar) en Camanchaca, una de las principales empresas de acuicultura en Chile, generalmente se encuentra en un rango de **4 a 6 kilogramos**. Este peso es considerado ideal para maximizar la calidad del producto y la rentabilidad en los mercados internacionales, especialmente en mercados como Estados Unidos, Europa y Asia.

Sin embargo, el peso de cosecha puede variar dependiendo de factores como las condiciones de cultivo, la estrategia comercial de la empresa y las demandas específicas del mercado. Para obtener información más precisa y actualizada, te recomendaría consultar directamente los informes anuales o de sostenibilidad de Camanchaca.

--- CON RAG (usando base de conocimiento de Camanchaca) ---
Consulta: ¿Cuál es el peso de cosecha objetivo para el salmón Atlántico en Camanchaca?

El peso de cosecha objetivo para el salmón A